# BirdCLEF+ 2026 — Random Forest Baseline
**Goal:** Identify 234 species from 5-second audio windows in Pantanal soundscapes.

**Eval Metric:** Macro-averaged ROC-AUC (skipping classes with no true positives).

**Pipeline overview:**
1. Load train.csv + taxonomy + train_soundscapes_labels
2. Extract log-mel spectrogram features from training audio (train_audio + labeled train_soundscapes)
3. Train one-vs-rest Random Forest with 5-fold OOF
4. Predict on test_soundscapes in 5-second windows
5. Write submission.csv

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, gc, warnings, time
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm.auto import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE        = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE / 'train_audio'
TRAIN_SND   = BASE / 'train_soundscapes'
TEST_SND    = BASE / 'test_soundscapes'
OUT         = Path('/kaggle/working')

# ── Audio config ──────────────────────────────────────────────────────────────
SR          = 32_000   # sample rate used in competition
SEGMENT_SEC = 5        # competition window = 5 s
N_MELS      = 64       # mel bands  (keep low → small feature vector)
HOP_LENGTH  = 512
N_FFT       = 1024
FMIN        = 50
FMAX        = 14_000

# ── Training config ───────────────────────────────────────────────────────────
# To keep runtime inside 90 min, we cap clips per species from train_audio.
# Increase MAX_CLIPS_PER_SPECIES if you want more data (costs more time).
MAX_CLIPS_PER_SPECIES = 30   # set None to use all (>16 GiB → will TLE)
MIN_RATING            = 3.0  # skip low-quality XC clips (0 = no rating → keep)
N_FOLDS               = 5
RF_N_ESTIMATORS       = 200
RF_MAX_DEPTH          = None  # full depth; reduce to 20 for speed
RF_N_JOBS             = -1
RANDOM_STATE          = 42

print('Config')

## 1 · Load Metadata

In [ ]:
train_df   = pd.read_csv(BASE / 'train.csv')
taxonomy   = pd.read_csv(BASE / 'taxonomy.csv')
snd_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

# The 234 species we must predict (column names in submission)
SPECIES = [c for c in sample_sub.columns if c != 'row_id']
print(f'Species to predict: {len(SPECIES)}')
print(f'Train audio clips : {len(train_df)}')
print(f'Soundscape labels : {len(snd_labels)}')
print(train_df.head(2))

## 2 · Feature Extraction Helper

In [ ]:
def audio_to_features(y, sr=SR):
    """
    Convert a raw waveform (one 5-second segment) to a 1-D feature vector.
    Features: mean + std of each mel band  →  2 * N_MELS values.
    Very fast, good enough for a Random Forest baseline.
    """
    # Pad / trim to exactly SEGMENT_SEC
    target_len = SR * SEGMENT_SEC
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)

    # Aggregate: mean & std per mel band
    feat = np.concatenate([log_mel.mean(axis=1), log_mel.std(axis=1)])
    return feat.astype(np.float32)


def load_clip_features(filepath, sr=SR):
    """Load an ogg/wav file, return features for first SEGMENT_SEC only."""
    try:
        y, _ = librosa.load(filepath, sr=sr, mono=True,
                            duration=SEGMENT_SEC)
        return audio_to_features(y, sr)
    except Exception:
        return None


FEAT_DIM = 2 * N_MELS
print(f'Feature dimension per segment: {FEAT_DIM}')

## 3 · Build Training Set from `train_audio` (XC + iNat)

In [ ]:
# Keep only species that appear in the 234-class submission
target_set = set(SPECIES)
df = train_df[train_df['primary_label'].isin(target_set)].copy()

# Filter by rating: keep 0 (no rating / iNat) and >= MIN_RATING
df = df[(df['rating'] == 0) | (df['rating'] >= MIN_RATING)].copy()

# Cap clips per species to control runtime
if MAX_CLIPS_PER_SPECIES:
    df = (
        df.groupby('primary_label', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), MAX_CLIPS_PER_SPECIES),
                                    random_state=RANDOM_STATE))
    ).reset_index(drop=True)

print(f'Clips after filtering / capping: {len(df)}')
print(f'Unique species in train_audio  : {df["primary_label"].nunique()}')

In [ ]:
X_audio, y_audio = [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc='train_audio feats'):
    fp = TRAIN_AUDIO / row['filename']
    if not fp.exists():
        continue
    feat = load_clip_features(fp)
    if feat is not None:
        X_audio.append(feat)
        y_audio.append(row['primary_label'])

X_audio = np.array(X_audio, dtype=np.float32)
y_audio = np.array(y_audio)
print(f'train_audio  X: {X_audio.shape}  unique labels: {len(set(y_audio))}')

## 4 · Add Labeled Train-Soundscape Segments

In [ ]:
def seconds_from_hms(hms_str):
    """Convert HH:MM:SS string to total seconds."""
    h, m, s = map(int, str(hms_str).split(':'))
    return h*3600 + m*60 + s


X_snd_list, y_snd_list = [], []

# Group labels by (filename, start) for efficiency
snd_labels['start_sec'] = snd_labels['start'].apply(seconds_from_hms)
snd_labels['end_sec']   = snd_labels['end'].apply(seconds_from_hms)

# Expand multi-label rows into one row per species
rows_expanded = []
for _, row in snd_labels.iterrows():
    for sp in str(row['primary_label']).split(';'):
        sp = sp.strip()
        if sp in target_set:
            rows_expanded.append((row['filename'], row['start_sec'], sp))

snd_expanded = pd.DataFrame(rows_expanded, columns=['filename','start_sec','species'])
print(f'Labeled soundscape segments (expanded): {len(snd_expanded)}')

In [ ]:
# Cache loaded soundscape files to avoid re-reading same file many times
_wav_cache = {}

def get_soundscape_segment(fname, start_sec):
    fp = TRAIN_SND / fname
    if not fp.exists():
        return None
    if fname not in _wav_cache:
        try:
            _wav_cache[fname], _ = librosa.load(fp, sr=SR, mono=True)
        except Exception:
            return None
    y_full = _wav_cache[fname]
    s = int(start_sec * SR)
    e = s + SR * SEGMENT_SEC
    segment = y_full[s:e]
    if len(segment) < SR:  # skip very short tail segments
        return None
    return segment


for _, row in tqdm(snd_expanded.iterrows(), total=len(snd_expanded),
                   desc='soundscape feats'):
    seg = get_soundscape_segment(row['filename'], row['start_sec'])
    if seg is not None:
        feat = audio_to_features(seg)
        X_snd_list.append(feat)
        y_snd_list.append(row['species'])

# Free the wav cache
_wav_cache.clear()
gc.collect()

X_snd = np.array(X_snd_list, dtype=np.float32)
y_snd = np.array(y_snd_list)
print(f'Soundscape  X: {X_snd.shape}')

## 5 · Combine & Binarize Labels

In [ ]:
X_all = np.concatenate([X_audio, X_snd], axis=0)
y_all = np.concatenate([y_audio, y_snd], axis=0)

# Map species label → column index
species_to_idx = {sp: i for i, sp in enumerate(SPECIES)}
n_classes = len(SPECIES)

# Build integer target vector (single-label; soundscape segments may have
# multiple but we've already expanded them above → each row is one species)
y_int = np.array([species_to_idx.get(sp, -1) for sp in y_all])
valid = y_int >= 0
X_all = X_all[valid]
y_int = y_int[valid]

print(f'Final training matrix: {X_all.shape}')
print(f'Classes represented  : {len(np.unique(y_int))} / {n_classes}')

## 6 · 5-Fold OOF Training (macro ROC-AUC)

In [ ]:
oof_probs = np.zeros((len(X_all), n_classes), dtype=np.float32)

# We use StratifiedKFold on the single integer class label
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y_int)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──')
    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_int[tr_idx], y_int[val_idx]

    rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        n_jobs=RF_N_JOBS,
        random_state=RANDOM_STATE,
        class_weight='balanced',  # important: class imbalance is heavy
        min_samples_leaf=2,
    )
    rf.fit(X_tr, y_tr)

    # predict_proba gives shape (n_val, n_classes_seen_in_train)
    # rf.classes_ tells us which integer classes were seen
    proba = rf.predict_proba(X_val)   # (n_val, len(rf.classes_))

    # Fill into full n_classes matrix
    fold_oof = np.zeros((len(val_idx), n_classes), dtype=np.float32)
    for j, cls in enumerate(rf.classes_):
        fold_oof[:, cls] = proba[:, j]
    oof_probs[val_idx] = fold_oof

    # --- OOF ROC-AUC (macro, skip classes with no true positive) ---
    # Build one-hot ground truth for val
    y_val_oh = np.zeros((len(y_val), n_classes), dtype=np.int8)
    y_val_oh[np.arange(len(y_val)), y_val] = 1

    # Compute per-class AUC only for classes that appear in val
    aucs = []
    for c in range(n_classes):
        if y_val_oh[:, c].sum() == 0:
            continue
        try:
            auc = roc_auc_score(y_val_oh[:, c], fold_oof[:, c])
            aucs.append(auc)
        except Exception:
            pass
    fold_auc = np.mean(aucs) if aucs else 0.0
    fold_aucs.append(fold_auc)
    print(f'  OOF macro-ROC-AUC (fold {fold+1}): {fold_auc:.4f}  [{len(aucs)} classes scored]')

print(f'\n=== Mean OOF macro-ROC-AUC: {np.mean(fold_aucs):.4f} ===')

In [ ]:
# Save OOF predictions for analysis
oof_df = pd.DataFrame(oof_probs, columns=SPECIES)
oof_df['true_label'] = [SPECIES[i] for i in y_int]
oof_df.to_csv(OUT / 'oof_predictions.csv', index=False)
print('OOF saved')

## 7 · Train Final Model on ALL Data

In [ ]:
print('Training final RF on full training set...')
rf_final = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    n_jobs=RF_N_JOBS,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    min_samples_leaf=2,
)
rf_final.fit(X_all, y_int)
print('Final model trained ✓')

# Map rf_final.classes_ → column indices for inference
final_cls = rf_final.classes_  # shape (n_seen,)
print(f'Classes seen by final model: {len(final_cls)}')

## 8 · Inference on Test Soundscapes

In [ ]:
def predict_soundscape(filepath, rf_model, cls_list):
    """
    Split a 1-minute soundscape into 5-second windows,
    extract features, predict, return list of (row_id, probs_array).
    """
    fname = Path(filepath).stem   # e.g. BC2026_Test_0001_S05_20250227_010002
    try:
        y_full, _ = librosa.load(filepath, sr=SR, mono=True)
    except Exception:
        return []

    results = []
    duration = len(y_full) / SR         # seconds (typically 60)
    n_segments = int(duration // SEGMENT_SEC)

    for seg_i in range(n_segments):
        start = seg_i * SEGMENT_SEC
        end   = start + SEGMENT_SEC
        end_time = int(end)             # submission uses end time in seconds
        row_id = f'{fname}_{end_time}'

        segment = y_full[int(start * SR): int(end * SR)]
        feat = audio_to_features(segment).reshape(1, -1)

        raw = rf_model.predict_proba(feat)[0]   # (n_seen_classes,)

        # Expand to full 234-class vector
        probs = np.full(n_classes, 1e-4, dtype=np.float32)  # tiny baseline
        for j, cls in enumerate(cls_list):
            probs[cls] = raw[j]

        results.append((row_id, probs))

    return results


# Discover test files
test_files = sorted(TEST_SND.glob('*.ogg'))
print(f'Test soundscapes found: {len(test_files)}')

In [ ]:
all_rows = []

for fp in tqdm(test_files, desc='Inferring test soundscapes'):
    rows = predict_soundscape(fp, rf_final, final_cls)
    all_rows.extend(rows)

print(f'Total prediction rows: {len(all_rows)}')

## 9 · Build & Save Submission

In [ ]:
row_ids = [r[0] for r in all_rows]
probs   = np.stack([r[1] for r in all_rows], axis=0)

sub = pd.DataFrame(probs, columns=SPECIES)
sub.insert(0, 'row_id', row_ids)

# Align to sample_submission (ensures correct column order & covers any missing rows)
sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')

# Fill any NaN rows (missing test file) with uniform baseline
baseline_prob = 1.0 / n_classes
sub[SPECIES] = sub[SPECIES].fillna(baseline_prob)

submission_path = OUT / 'submission.csv'
sub.to_csv(submission_path, index=False)
print(f'submission.csv saved  →  shape {sub.shape}')
print(sub.head(3))

In [ ]:
# Quick sanity checks
assert list(sub.columns) == list(sample_sub.columns), 'Column mismatch!'
assert len(sub) == len(sample_sub), f'Row count mismatch: {len(sub)} vs {len(sample_sub)}'
assert sub[SPECIES].isnull().sum().sum() == 0, 'NaN in predictions!'
print('All sanity checks passed...')
print(f'OOF mean macro-ROC-AUC = {np.mean(fold_aucs):.4f}')